In [26]:
# Boolean Query Generation R&D
# This notebook is for experimenting with different prompt variations for generating boolean queries

import sys
import asyncio
from pathlib import Path

# Add backend to path (go up 3 levels: boolean_queries -> r_and_d -> testing -> backend)
backend_path = Path.cwd().parent.parent.parent
sys.path.insert(0, str(backend_path))

# Load environment variables from .env file
from dotenv import load_dotenv
load_dotenv(backend_path / ".env")

# Now import app modules
from app.services.analysis.references import ReferencesService
from app.services import openalex
from app.core.config import settings
from openai import AsyncOpenAI
from app.services.analysis.prompts import BOOLEAN_QUERY_SYSTEM_PROMPT
import tiktoken

import pandas as pd
df = pd.read_csv("inputs/reference_queries.csv")
df

,identifier,question,boolean_query,boolean_query_reviews
0,reference_0,What is the effect of free school meals on obesity and calorie consumption?,"(school AND (meal OR dinner OR food)) AND ((""Energy Intake"" OR ""Calorie consumption"" OR Calori* OR ""Calories consumed"" OR ""Calorie intake"" OR ""Caloric intake"" OR Energy OR ""Food consum*"" OR kcal OR kj) OR (Obesity OR overweight OR ""over-weight"" OR Weight OR BMI OR ""body composition""))","(school AND (meal OR dinner OR food)) AND ((""Energy Intake"" OR ""Calorie consumption"" OR Calori* OR ""Calories consumed"" OR ""Calorie intake"" OR ""Caloric intake"" OR Energy OR ""Food consum*"" OR kcal OR kj) OR (Obesity OR overweight OR ""over-weight"" OR Weight OR BMI OR ""body composition"")) AND (systematic* OR ""meta-analys*"" OR ""narrative synthes*"")"
1,reference_1,What is the effect of restricting advertising of unhealthy food and drink on calorie consumption or obesity outcomes in adults or children?,"(Advert* OR Marketing OR Commercial* OR Promotion*) AND (Food* OR Foodstuff* OR Snack* OR Nutrition* OR Diet OR ""Food and Beverages"" OR ""Food Preferences"" OR ""Diet, Food, and Nutrition"" OR ""Fast Foods"") AND ((""Calorie consumption"" OR Calori* OR ""Calories consumed"" OR ""Calorie intake"" OR ""Caloric intake"" OR Energy OR ""Energy Intake"" OR ""Food consumption"" OR ""Food consumed"" OR Weight OR BMI OR ""weight loss"" OR kcal OR kj) OR (obesity OR overweight OR ""over-weight"" OR BMI OR ""body weight"" OR bodyweight OR ""Body mass index""))","(Advert* OR Marketing OR Commercial* OR Promotion*) AND (Food* OR Foodstuff* OR Snack* OR Nutrition* OR Diet OR ""Food and Beverages"" OR ""Food Preferences"" OR ""Diet, Food, and Nutrition"" OR ""Fast Foods"") AND ((""Calorie consumption"" OR Calori* OR ""Calories consumed"" OR ""Calorie intake"" OR ""Caloric intake"" OR Energy OR ""Energy Intake"" OR ""Food consumption"" OR ""Food consumed"" OR Weight OR BMI OR ""weight loss"" OR kcal OR kj) OR (obesity OR overweight OR ""over-weight"" OR BMI OR ""body weight"" OR bodyweight OR ""Body mass index"")) AND (""systematic review"" OR systematic* OR ""meta-analys*"" OR ""narrative synthes*"")"
2,reference_2,"What is the effect of volume promotions on purchasing, obesity and calorie consumption","(multipack* OR ""multi-buy"" OR multibuy OR ""multi-unit"" OR multiunit OR ""multi unit"" OR ""multi buy"" OR ""multi pack"" OR ""buy one get one free"" OR ""buy-one-get-one-free"" OR ""three for two"" OR ""3 for 2"" OR ""combination offer"" OR ""linked offer"" OR ""volume promotion"" OR ""volume discount"") AND ((""Food and Beverages"" OR ""Food Preferences"" OR ""Diet, Food, and Nutrition"" OR ""Fast Foods"" OR Food* OR Foodstuff* OR Snack* OR Nutrition* OR Diet) OR (purchas* OR buy OR buying OR sale*) OR (""Energy Intake"" OR ""Calorie consumption"" OR Calori* OR ""Calories consumed"" OR ""Calorie intake"" OR ""Caloric intake"" OR Energy OR ""Food consum*"" OR kcal OR kj)) AND (Obesity OR overweight OR ""over-weight"" OR Weight OR BMI OR ""body composition"")","(multipack* OR ""multi-buy"" OR multibuy OR ""multi-unit"" OR multiunit OR ""multi unit"" OR ""multi buy"" OR ""multi pack"" OR ""buy one get one free"" OR ""buy-one-get-one-free"" OR ""three for two"" OR ""3 for 2"" OR ""combination offer"" OR ""linked offer"" OR ""volume promotion"" OR ""volume discount"") AND ((""Food and Beverages"" OR ""Food Preferences"" OR ""Diet, Food, and Nutrition"" OR ""Fast Foods"" OR Food* OR Foodstuff* OR Snack* OR Nutrition* OR Diet) OR (purchas* OR buy OR buying OR sale*) OR (""Energy Intake"" OR ""Calorie consumption"" OR Calori* OR ""Calories consumed"" OR ""Calorie intake"" OR ""Caloric intake"" OR Energy OR ""Food consum*"" OR kcal OR kj)) AND (Obesity OR overweight OR ""over-weight"" OR Weight OR BMI OR ""body composition"") AND (systematic* OR ""meta-analys*"" OR ""narrative synthesis"")"
3,reference_3,Does universal breastfeeding information provision improve health outcomes for mothers

In [ ]:
# pd.set_option('display.max_colwidth', None)
# # df = df.assign(**{"ID": "reference_" + df.index.astype(str)})
# df = (
#     df
#     .rename({
#         "Topic": "topic",
#         "Research Question": "question",
#         "Policy Atlas Custom Boolean": "boolean_query_reviews",
#         "AVG Relevant Retreival Rate": "avg_relevant_retrieval_rate",
#         "ID": "identifier",
#     }, axis=1)
#     .assign(**{
#         "boolean_query": lambda df: df["boolean_query_reviews"].apply(lambda x: " AND ".join(x.split("AND")[0:-1]))
#     })
# )
# # df.to_csv("inputs/reference_queries.csv", index=False)
# df = df[["identifier", "question", "boolean_query", "boolean_query_reviews"]]
# df.to_csv("inputs/reference_queries.csv", index=False)


In [18]:
df["boolean_query_reviews"].str.split("AND").iloc[0][0:-1]

AttributeError: 'list' object has no attribute 'str'

In [26]:
import yaml

config = yaml.load(open('config.yaml', 'r'), Loader=yaml.FullLoader)

In [22]:
import importlib
importlib.reload(openalex);

In [ ]:
reference_boolean_queries = df["Policy Atlas Custom Boolean"].tolist()
test_query = df["Research Question"].iloc[0]
print(test_query)
print(boolean_query)

What is the effect of free school meals on obesity and calorie consumption?
(school AND (meal OR dinner OR food)) AND (("Energy Intake" OR "Calorie consumption" OR Calori* OR "Calories consumed" OR "Calorie intake" OR "Caloric intake" OR Energy OR "Food consum*" OR kcal OR kj) OR (Obesity OR overweight OR "over-weight" OR Weight OR BMI OR "body composition")) AND (systematic* OR "meta-analys*" OR "narrative synthes*")


In [14]:
# df["Policy Atlas Custom Boolean"]

In [23]:
# Search OpenAlex with the generated boolean query
openalex_service = openalex.OpenAlexService()

# Fetch results
results_df, n_total = await openalex_service.search_minimal(
    query=boolean_query,
    max_results=200,  # Adjust as needed
    min_citations=0,
    return_n_total=True,
)

In [24]:
pd.set_option('display.max_colwidth', None)
results_df

,id,doi,title,cited_by_count
0,https://openalex.org/W1534146668,https://doi.org/10.1111/obr.12142,A systematic review of the influence of the retail food environment around schools on obesity‐related outcomes,177
1,https://openalex.org/W1744259402,https://doi.org/10.1111/obr.12224,Effect of changes to the school food environment on eating behaviours and/or body weight in children: a systematic review,170
2,https://openalex.org/W3001945172,https://doi.org/10.1093/nutrit/nuz110,Retail food environment around schools and overweight: a systematic review,56
3,https://openalex.org/W4311581251,https://doi.org/10.1111/obr.13536,Association of fast‐food restaurants with overweight and obesity in school‐aged children and adolescents: A systematic review and meta‐analysis,26
4,https://openalex.org/W2788612692,https://doi.org/10.1093/pubmed/fdy048,The impact of hot food takeaways near schools in the UK on childhood obesity: a systematic review of the evidence,24
...,...,...,...,...
195,https://openalex.org/W2115317190,https://doi.org/10.1111/j.1399-5448.2009.00567.x,Exercise in children and adolescents with diabetes,164
196,https://openalex.org/W4390690812,https://doi.org/10.1093/heapro/daad194,Unpacking the cost of the lunchbox for Australian families: a secondary analysis,9
197,https://openalex.org/W4249548800,https://doi.org/10.1596/0-8213-5769-7,Targeting of Transfers in Developing Countries,295
198,https://openalex.org/W3014107638,https://doi.org/10.26505/djm.18014900828,Obesity in Primary Schools Children in Baquba City,3


In [52]:
len(results_df)

2

In [37]:
# # Custom prompt for experimentation
# # You can modify this to test different variations
# CUSTOM_BOOLEAN_PROMPT = """You are an expert at creating boolean search queries for academic literature databases like OpenAlex and Overton.

# Given a research question, extract the key concepts and create a targeted boolean search query. DO NOT use the entire research question as a search term.

# IMPORTANT: Break down the research question into its core components and search terms. For example:
# - Research question: "What is the biggest interventions for decarbonising home heating?"
# - Key concepts: decarbonisation, home heating, interventions, residential heating, carbon reduction
# - Boolean query: (decarbonis* OR "carbon reduction" OR "emissions reduction") AND ("home heating" OR "residential heating" OR "domestic heating") AND (intervention* OR program* OR policy OR measure*)

# For policy-specific queries, focus on the underlying research topics rather than specific policy names:
# - Research question: "Which UK home-heating incentives have reduced gas?"
# - Key concepts: heating policy evaluation, residential gas consumption, energy efficiency programs
# - Boolean query: ("residential heating" OR "home heating" OR "domestic heating") AND ("gas consumption" OR "natural gas" OR "gas demand") AND (policy OR program* OR incentive* OR intervention*) AND (reduc* OR efficiency OR savings)

# Guidelines:
# 1. Extract 2-4 main concepts from the research question
# 2. Use AND to connect different concepts that all should be present in the documents
# 3. Use OR to include synonyms and related terms within each concept
# 4. Use wildcards (*) for word variations (e.g., intervention*)
# 5. Use quotes for exact phrases when beneficial
# 6. Include both technical and common language terms
# 7. Focus on terms that would realistically appear in academic paper titles and abstracts
# 8. For policy queries, focus on research about the underlying phenomena rather than specific policy names
# 9. Consider broader academic terminology (evaluation, effectiveness, impact, outcomes)
# 10. Prioritize nouns and key descriptive terms over question words (what, how, why)
# 11. Include related research terms like "evaluation", "impact", "effectiveness" for policy questions

# Most importantly, keep the query sufficiently general so that we get more results, but roughly in the right ballpark.

# Return ONLY the boolean query string, nothing else."""


In [73]:
settings.LLM_MODEL

'gpt-4o-mini'

In [74]:
models = [
    'gpt-4o',
    'gpt-4o-mini',
    'gpt-4.1',
    'gpt-4.1-mini',
    'gpt-4.1-nano',
    'gpt-5',
    'gpt-5-mini',
    'gpt-5-nano',
]

In [ ]:
# Function to generate boolean query with custom prompt
async def call_llm(
    user_message: str,
    system_prompt: str,
    model: str = settings.LLM_MODEL,
    temperature: float = 0.0,
    max_tokens: int = 1000,
) -> str:
    """Generate a boolean query using a custom prompt."""
    client = AsyncOpenAI(api_key=settings.OPENAI_API_KEY)
    
    if model not in ['gpt-5', 'gpt-5-mini', 'gpt-5-nano']:
        resp = await client.chat.completions.create(
            model=model,
            temperature=temperature,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_message},
            ],
            max_tokens=max_tokens,
        )
    else:
        resp = await client.chat.completions.create(
            model=model,
            messages=[
                {"role": "user", "content": system_prompt},
                {"role": "user", "content": user_message}],
        )
    
    boolean_query = resp.choices[0].message.content.strip()
    return boolean_query


In [85]:
# Run a test with all the models
boolean_queries = []
for model in models:
    boolean_query = await generate_boolean_query_custom(test_query, BOOLEAN_QUERY_SYSTEM_PROMPT, model=model)
    boolean_queries.append(boolean_query)
    print(model)
    print(boolean_query)
    print("\n")


gpt-4o
("free school meals" OR "school lunch program" OR "school meal program") AND (obesity OR "body weight" OR "weight gain") AND ("calorie consumption" OR "caloric intake" OR "dietary intake")


gpt-4o-mini
("free school meals" OR "school meal programs" OR "school nutrition") AND (obesity OR "weight gain" OR "overweight") AND ("calorie consumption" OR "caloric intake" OR "energy intake") AND (effect* OR impact OR evaluation OR outcomes)


gpt-4.1
("free school meals" OR "school meal program*" OR "universal school meals") AND (obesity OR "body mass index" OR BMI OR "overweight" OR "weight status") AND ("calorie consumption" OR "energy intake" OR "dietary intake" OR "nutrition") AND (effect* OR impact* OR outcome* OR evaluation)


gpt-4.1-mini
("free school meals" OR "school meal program*" OR "school feeding") AND (obes* OR "body mass index" OR BMI OR "weight gain") AND ("calorie consumption" OR "energy intake" OR "dietary intake" OR "food intake") AND (effect* OR impact OR outcome* O

In [ ]:
n_totals = []
for boolean_query in boolean_queries:
    # Fetch results
    _, n_total = await openalex_service.search_minimal(
        query=boolean_query,
        max_results=None,  # Adjust as needed
        min_citations=0,
        return_n_total=True,
    )
    n_totals.append(n_total)

In [87]:
res_df = pd.DataFrame({
    'model': models,
    'boolean_query': boolean_queries,
    'n_total': n_totals
})
res_df

,model,boolean_query,n_total
0,gpt-4o,"(""free school meals"" OR ""school lunch program"" OR ""school meal program"") AND (obesity OR ""body weight"" OR ""weight gain"") AND (""calorie consumption"" OR ""caloric intake"" OR ""dietary intake"")",27
1,gpt-4o-mini,"(""free school meals"" OR ""school meal programs"" OR ""school nutrition"") AND (obesity OR ""weight gain"" OR ""overweight"") AND (""calorie consumption"" OR ""caloric intake"" OR ""energy intake"") AND (effect* OR impact OR evaluation OR outcomes)",9
2,gpt-4.1,"(""free school meals"" OR ""school meal program*"" OR ""universal school meals"") AND (obesity OR ""body mass index"" OR BMI OR ""overweight"" OR ""weight status"") AND (""calorie consumption"" OR ""energy intake"" OR ""dietary intake"" OR ""nutrition"") AND (effect* OR impact* OR outcome* OR evaluation)",48
3,gpt-4.1-mini,"(""free school meals"" OR ""school meal program*"" OR ""school feeding"") AND (obes* OR ""body mass index"" OR BMI OR ""weight gain"") AND (""calorie consumption"" OR ""energy intake"" OR ""dietary intake"" OR ""food intake"") AND (effect* OR impact OR outcome* OR association*)",11
4,gpt-4.1-nano,"(""free school meals"" OR ""school meal programs"" OR ""school feeding"") AND (obesity OR ""body weight"" OR ""overweight"") AND (""calorie consumption"" OR ""caloric intake"" OR ""energy intake"") AND (effect* OR impact OR outcome* OR evaluat*)",4
5,gpt-5,"((""free"" OR ""no-cost"" OR ""no cost"" OR universal) AND (""school meal*"" OR ""school lunch*"" OR ""school breakfast*"" OR ""school feeding program*"")) AND (obes* OR overweight OR ""body mass index"" OR BMI OR ""body weight"" OR adiposit* OR ""weight status"" OR ""calorie*"" OR ""caloric intake"" OR ""energy intake"" OR ""dietary intake"" OR ""food consumption"" OR ""energy consumption"" OR kcal) AND (effect* OR impact* OR evaluation OR effectiveness OR outcome* OR ""difference-in-difference*"" OR experiment* OR trial*)",218
6,gpt-5-mini,"((""free school meal*"" OR ""free school lunch*"" OR ""school meal*"" OR ""school lunch program*"" OR ""school feeding program*"" OR ""school feeding"") AND (obes* OR overweight OR ""body mass index"" OR BMI OR adiposit* OR ""body weight"" OR ""weight gain"") AND (calor* OR ""energy intake"" OR ""caloric intake"" OR ""dietary intake"" OR ""food intake"" OR kcal OR ""energy consumption"") AND (effect* OR impact* OR evaluat* OR associat* OR outcome* OR random* OR RCT OR ""natural experiment"" OR quasi-experiment* OR intervent* OR policy OR program*) AND (child* OR adolescen* OR student* OR pupil* OR schoolchild*))",48
7,gpt-5-nano,"((""free school meals"" OR ""school meals"" OR ""school lunch program"" OR ""pupil meal"" OR ""meal program"") AND (obesity OR ""body mass index"" OR BMI OR overweight OR adiposity OR ""obesity prevalence"") AND (calor* intake OR ""calorie intake"" OR ""energy intake"" OR ""dietary energy"" OR ""dietary intake"") AND (impact* OR effect* OR evaluat* OR effectiveness OR outcome*))",0


In [69]:
# Generate boolean query
boolean_query = await generate_boolean_query_custom(test_query, BOOLEAN_QUERY_SYSTEM_PROMPT)


In [70]:
boolean_query

'("free school meals" OR "school meal programs" OR "school nutrition") AND (obesity OR "weight gain" OR "overweight") AND ("calorie consumption" OR "caloric intake" OR "energy intake") AND (effect* OR impact OR evaluation OR outcomes)'

In [ ]:
# Fetch results
results_df, n_total = await openalex_service.search_minimal(
    query=boolean_query,
    max_results=1,  # Adjust as needed
    min_citations=0,
    return_n_total=True,
)

print(f"\n✅ Retrieved {len(results_df)} results from OpenAlex")



✅ Retrieved 9 results from OpenAlex


In [57]:
n_total

9

In [ ]:
# Inspect specific results
# Show full titles and abstracts for manual relevance assessment
for idx, row in results_df.head(10).iterrows():
    print(f"\n{'='*80}")
    print(f"📄 {row['title']}")
    print(f"📅 Year: {row['publication_year']} | 📚 Citations: {row['cited_by_count']}")
    print(f"📝 Abstract: {row['abstract'][:300]}...")
    print(f"🔗 {row['landing_page_url']}")


In [1]:
# Helper function to compare multiple prompts
async def compare_prompts(natural_query: str, prompts_dict: dict, max_results: int = 50):
    """
    Compare multiple prompt variations and their results.
    
    Args:
        natural_query: The natural language query
        prompts_dict: Dictionary of {prompt_name: prompt_text}
        max_results: Max results to fetch from OpenAlex
    
    Returns:
        Dictionary of results for each prompt
    """
    openalex = OpenAlexService()
    results = {}
    
    for name, prompt in prompts_dict.items():
        print(f"\n{'='*80}")
        print(f"Testing prompt: {name}")
        print(f"{'='*80}")
        
        # Generate boolean query
        boolean_query = await generate_boolean_query_custom(natural_query, prompt)
        
        # Fetch results
        df = await openalex.search(
            query=boolean_query,
            max_results=max_results,
            min_citations=settings.DEFAULT_MIN_CITATIONS,
        )
        
        results[name] = {
            'boolean_query': boolean_query,
            'results_df': df,
            'count': len(df)
        }
        
        print(f"✅ Retrieved {len(df)} results\n")
    
    return results


In [ ]:
# Example: Compare the current prompt with a variation
# Uncomment and modify to test different prompts

# prompts_to_compare = {
#     'current': CUSTOM_BOOLEAN_PROMPT,
#     'variation_1': """Your modified prompt here...""",
# }

# comparison_results = await compare_prompts(test_query, prompts_to_compare)

# # Show comparison summary
# for name, data in comparison_results.items():
#     print(f"\n{name}:")
#     print(f"  Boolean query: {data['boolean_query']}")
#     print(f"  Results count: {data['count']}")
#     if data['count'] > 0:
#         print(f"  Avg citations: {data['results_df']['cited_by_count'].mean():.1f}")
#         print(f"  Year range: {data['results_df']['publication_year'].min()}-{data['results_df']['publication_year'].max()}")


In [ ]:
# Calculate basic statistics for results
def calculate_stats(df):
    """Calculate basic statistics for a results dataframe."""
    if len(df) == 0:
        return "No results"
    
    stats = {
        'total_results': len(df),
        'avg_citations': df['cited_by_count'].mean(),
        'median_citations': df['cited_by_count'].median(),
        'year_range': f"{df['publication_year'].min()}-{df['publication_year'].max()}",
        'open_access_pct': (df['is_oa'].sum() / len(df) * 100) if 'is_oa' in df.columns else None,
    }
    
    return stats

# Example usage
stats = calculate_stats(results_df)
print("\n📊 Results Statistics:")
for key, value in stats.items():
    print(f"  {key}: {value}")


In [ ]:
# Future: For calculating recall/precision metrics
# You can manually label results and calculate metrics

# Example structure for future eval:
# relevance_labels = {
#     'doc_id_1': 1,  # relevant
#     'doc_id_2': 0,  # not relevant
#     # ...
# }

# def calculate_precision_at_k(df, labels, k=10):
#     """Calculate precision@k for labeled results."""
#     top_k = df.head(k)
#     relevant_count = sum(labels.get(doc_id, 0) for doc_id in top_k['id'])
#     return relevant_count / k

print("✅ Notebook ready for experimentation!")
print("\nNext steps:")
print("1. Run the cells to generate a boolean query and fetch results")
print("2. Inspect the results for relevance")
print("3. Modify CUSTOM_BOOLEAN_PROMPT to test variations")
print("4. Use compare_prompts() to test multiple prompts at once")
print("5. Calculate statistics to evaluate different approaches")
